# Basic FF Neural Network Analysis for the effects of weather to a days delays

In [8]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [4]:
file_name = "../../data/delays/subway/cleaned_data/cleaned_subway_data.csv"
weather_file_name = "../../data/weather/Toronto_City_Centre_2XUGG (43.63, -79.4).csv"

sub_data = pd.read_csv(file_name)

weather_data = pd.read_csv(weather_file_name)

In [5]:
sub_data["Time"] = pd.to_datetime(sub_data["Datetime"])

In [6]:
sub_data["date_only"] = pd.to_datetime(sub_data["Time"])
sub_data["date_only"] = sub_data["date_only"].dt.floor("1D")
sub_data_copy = sub_data

date_range = pd.date_range(start="2023-01-01", end="2024-12-31", freq="1D")
dummy = pd.DataFrame({'date_only': date_range})

sub_data = dummy.merge(sub_data, on='date_only', how='left')

# drop outliers (display these later)
super_delays = sub_data[sub_data["Min Delay"]>60].dropna()
super_delays = super_delays[super_delays["Min Delay"]<1000]
sub_data = sub_data[sub_data["Min Delay"]<=60] 

mean_sub_data = sub_data.groupby("date_only")[["Min Delay", "Min Gap"]].mean().reset_index()
mean_sub_data = mean_sub_data[mean_sub_data["date_only"] >= "2023-01-01"]
count_sub_data = sub_data.groupby("date_only")[["Min Delay", "Min Gap"]].count().reset_index()
count_sub_data = count_sub_data[count_sub_data["date_only"] >= "2023-01-01"]
super_means = super_delays.groupby("date_only")[["Min Delay", "Min Gap"]].mean().reset_index()

display(mean_sub_data)

display(weather_data)

,date_only,Min Delay,Min Gap
0,2023-01-01,2.906977,5.558140
1,2023-01-02,2.926829,5.219512
2,2023-01-03,2.883721,4.604651
3,2023-01-04,3.754098,5.377049
4,2023-01-05,2.088889,3.488889
...,...,...,...
726,2024-12-27,1.688312,2.584416
727,2024-12-28,1.390244,2.524390
728,2024-12-29,2.218750,3.187500
729,2024-12-30,1.613636,2.659091


,date,tavg,tmin,tmax,prcp,wdir,wspd
0,2023-01-01 0:00:00,2.5,1.5,3.3,0.3,234.0,9.1
1,2023-01-02 0:00:00,2.5,-2.0,5.5,0.0,260.0,6.8
2,2023-01-03 0:00:00,2.3,0.0,3.5,1.3,57.0,14.5
3,2023-01-04 0:00:00,3.0,2.6,3.7,25.1,66.0,17.8
4,2023-01-05 0:00:00,2.8,0.5,4.1,0.5,216.0,11.1
...,...,...,...,...,...,...,...
726,2024-12-27 0:00:00,1.2,-0.2,2.5,0.0,87.0,25.2
727,2024-12-28 0:00:00,5.4,2.4,9.2,1.0,176.0,19.0
728,2024-12-29 0:00:00,5.0,0.6,10.3,27.0,81.0,22.0
729,2024-12-30 0:00:00,4.6,2.9,9.3,2.4,238.0,36.8


In [91]:
class FFN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(FFN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size) 
        self.fc2 = nn.Linear(hidden_size, hidden_size) 
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)

        return out

In [92]:
input_size = 6
output_size = 2 # mean min delay, count of delay
hidden_size = 64
batch_size = 8

In [93]:
model = FFN(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [98]:
X = weather_data[weather_data.columns[1:]]
y1 = mean_sub_data['Min Delay']
y2 = count_sub_data["Min Delay"]
good_rows = X.notna().all(axis=1)

X = X[good_rows]
y1 = y1[good_rows]
y2 = y2[good_rows]

X = torch.tensor(X.values, dtype = torch.float32)
y = torch.stack([torch.tensor(y1.values, dtype = torch.float32),torch.tensor(y2.values, dtype = torch.float32)], dim = 1)

X_train = X[:-200]
y_train = y[:-200]

X_test = X[-200:]
y_test = y[-200:]

print(X.shape)
print(y.shape)

print(torch.isnan(X).any())
print(torch.isnan(y).any())

torch.Size([719, 6])
torch.Size([719, 2])
tensor(False)
tensor(False)


In [99]:
num_epochs = 50

for epoch in range(num_epochs):
    for i in range(0, len(X_train), batch_size):
        x_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch: {epoch}, MSE: {loss}")

Epoch: 0, MSE: 138.38075256347656
Epoch: 1, MSE: 160.76492309570312
Epoch: 2, MSE: 163.3853759765625
Epoch: 3, MSE: 160.711669921875
Epoch: 4, MSE: 158.72238159179688
Epoch: 5, MSE: 159.70675659179688
Epoch: 6, MSE: 160.20716857910156
Epoch: 7, MSE: 156.59019470214844
Epoch: 8, MSE: 152.36585998535156
Epoch: 9, MSE: 151.02685546875
Epoch: 10, MSE: 149.35244750976562
Epoch: 11, MSE: 146.5003662109375
Epoch: 12, MSE: 145.65170288085938
Epoch: 13, MSE: 142.5589599609375
Epoch: 14, MSE: 142.96812438964844
Epoch: 15, MSE: 140.068359375
Epoch: 16, MSE: 139.37229919433594
Epoch: 17, MSE: 135.42625427246094
Epoch: 18, MSE: 137.05958557128906
Epoch: 19, MSE: 132.01699829101562
Epoch: 20, MSE: 133.3795166015625
Epoch: 21, MSE: 133.14292907714844
Epoch: 22, MSE: 129.24058532714844
Epoch: 23, MSE: 130.2848663330078
Epoch: 24, MSE: 126.59777069091797
Epoch: 25, MSE: 125.58745574951172
Epoch: 26, MSE: 123.14266204833984
Epoch: 27, MSE: 123.09342956542969
Epoch: 28, MSE: 119.13629913330078
Epoch: 29,

In [100]:
model.eval() 
with torch.no_grad():
    predictions = model(X_test) 

mse = criterion(predictions, y_test.long())
print(f"Cross Entropy on test set: {mse.item()}")

Cross Entropy on test set: 142.31861877441406
